# Day 21 — Voltage inheritance for untagged OSM lines

Phase 1A's coverage audit found **226 transmission-shaped OSM lines** with
unparseable `voltage` tags (Samar 230 km, Eastern Samar 166 km, Leyte 73 km,
Cebu 27 km). Phase 1B's voltage parser dropped them. This notebook recovers
the ones that can be confidently re-attached: for each untagged candidate,
snap both endpoints to the nearest *tagged* transmission bus within
`MAX_SNAP_KM = 1.5 km`. If both ends match different buses of compatible
voltage, inherit the voltage and add the line.

**Conservative by design.** Trivial short candidates (< 0.3 km, mostly
intra-substation jumpers) are filtered out; self-loops (both ends → same
bus) are excluded; mixed-voltage matches take the max (interpreted as a
step-down tap to the lower side).

**Position in pipeline:** runs after `11_substation_merge.ipynb` so that
bus IDs are final (substation merges won't later rewrite the new lines'
endpoints). Wired into `scripts/run_phase1.py`.

**Output:** appends recovered rows to `backend/data/processed/lines.csv`
with `data_source = 'osm_inferred_voltage'`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point

RAW  = Path('../backend/data/raw/visayas_power_raw.geojson')
PROC = Path('../backend/data/processed')

MAX_SNAP_KM = 1.5            # endpoint → bus match radius
MIN_LINE_KM = 0.3            # filter trivial jumpers
TRANSMISSION_MIN_KV = 33.0   # subtransmission and up

# Canonical per-voltage line parameters (overhead). Matches values used
# elsewhere in the pipeline for OSM and synthetic_v1 lines.
IMPEDANCE = {
    60.0:  {'r': 0.180, 'x': 0.420, 'i': 0.450},
    69.0:  {'r': 0.150, 'x': 0.400, 'i': 0.500},
    138.0: {'r': 0.080, 'x': 0.400, 'i': 0.700},
    230.0: {'r': 0.050, 'x': 0.400, 'i': 0.900},
    350.0: {'r': 0.0754, 'x': 0.121, 'i': 0.645},  # HVDC
}

## §1 — Identify candidate untagged lines

In [2]:
def parse_voltage_kv(v) -> list[int]:
    """Same parser as notebook 05 — returns plausible kV values."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    out = []
    for part in re.split(r'[;,/]', str(v)):
        m = re.search(r'\d+', part)
        if not m:
            continue
        n = int(m.group())
        kv = n / 1000 if n >= 1000 else n
        if 0.4 <= kv <= 1000:
            out.append(int(round(kv)))
    return out

osm = gpd.read_file(RAW)
lines_osm = osm[osm['power'].isin(['line', 'cable'])].copy()
lines_osm = lines_osm[lines_osm.geometry.geom_type.isin(['LineString', 'MultiLineString'])]
lines_osm['vlist'] = lines_osm['voltage'].apply(parse_voltage_kv)
untagged = lines_osm[lines_osm['vlist'].apply(len) == 0].copy()

untagged_m = untagged.to_crs(32651)
untagged_m['length_km'] = untagged_m.geometry.length / 1000
untagged_m = untagged_m[untagged_m['length_km'] >= MIN_LINE_KM].copy()
untagged_m['is_cable'] = untagged_m['power'] == 'cable'

print(f'Raw OSM features:                  {len(osm)}')
print(f'Lines/cables with line geometry:   {len(lines_osm)}')
print(f'Untagged voltage (initial):        {len(untagged)}')
print(f'Candidate set (≥ {MIN_LINE_KM} km):       {len(untagged_m)}')
print(f'  of which power=cable:            {untagged_m["is_cable"].sum()}')

Raw OSM features:                  19607
Lines/cables with line geometry:   631
Untagged voltage (initial):        226
Candidate set (≥ 0.3 km):       82
  of which power=cable:            7


/Users/julius/polymath/Projects/powergrid/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 1237981974 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


## §2 — Snap endpoints to tagged transmission buses

Builds a UTM-51N point GeoDataFrame of every tagged tx bus, indexes it,
then for each candidate finds the nearest bus to each endpoint within
`MAX_SNAP_KM`. Classification:

- `no_match` — at least one endpoint has no bus within tolerance
- `self_loop` — both endpoints snap to the same bus (intra-substation jumper)
- `matched_same_voltage` — both endpoints, different buses, same voltage
- `matched_mixed_voltage` — different buses, different voltages (tap line; inherit max)

In [3]:
buses = pd.read_csv(PROC / 'buses.csv')
tx = buses[(buses['voltage_kv'] >= TRANSMISSION_MIN_KV)
           & buses['voltage_kv'].notna()
           & buses['bus_type'].isin(['substation', 'tower', 'hvdc'])].copy()
tx_geom = gpd.GeoDataFrame(
    tx[['bus_id', 'voltage_kv']],
    geometry=gpd.points_from_xy(tx['lon'], tx['lat'], crs='EPSG:4326'),
).to_crs(32651)
tx_sindex = tx_geom.sindex
print(f'Tagged tx buses: {len(tx_geom)}')
print('  by voltage:', tx['voltage_kv'].value_counts().to_dict())

def snap(point):
    buf = point.buffer(MAX_SNAP_KM * 1000)
    cand = tx_geom.iloc[list(tx_sindex.query(buf))]
    if len(cand) == 0:
        return None, None, None
    d = cand.geometry.distance(point)
    i = d.idxmin()
    if d.loc[i] > MAX_SNAP_KM * 1000:
        return None, None, None
    return tx_geom.loc[i, 'bus_id'], float(tx_geom.loc[i, 'voltage_kv']), float(d.loc[i])

records = []
for idx, row in untagged_m.iterrows():
    geom = row.geometry
    if geom.geom_type == 'MultiLineString':
        segs = list(geom.geoms)
        p_start, p_end = Point(segs[0].coords[0]), Point(segs[-1].coords[-1])
    else:
        coords = list(geom.coords)
        p_start, p_end = Point(coords[0]), Point(coords[-1])
    b1, kv1, d1 = snap(p_start)
    b2, kv2, d2 = snap(p_end)
    if b1 is None or b2 is None:
        outcome, inferred = 'no_match', None
    elif b1 == b2:
        outcome, inferred = 'self_loop', None
    elif kv1 == kv2:
        outcome, inferred = 'matched_same_voltage', kv1
    else:
        outcome, inferred = 'matched_mixed_voltage', max(kv1, kv2)
    records.append(dict(
        osm_idx=idx, length_km=row['length_km'], is_cable=row['is_cable'],
        from_bus=b1, to_bus=b2, from_kv=kv1, to_kv=kv2,
        from_dist_m=d1, to_dist_m=d2, inferred_kv=inferred, outcome=outcome,
    ))

rec = pd.DataFrame(records)
print()
print('Outcome distribution:')
print(rec['outcome'].value_counts().to_string())

Tagged tx buses: 205
  by voltage: {138.0: 126, 69.0: 41, 230.0: 28, 350.0: 5, 60.0: 5}

Outcome distribution:
outcome
no_match                 58
matched_same_voltage     12
self_loop                11
matched_mixed_voltage     1


## §3 — Build line records and dedupe

Each recoverable candidate becomes one row in `lines.csv` with
`data_source = 'osm_inferred_voltage'`. Deduplicate against existing lines
with the same `{from_bus, to_bus}` set so we don't double-add a corridor
that survived the voltage parser via a sibling tag.

In [4]:
lines_existing = pd.read_csv(PROC / 'lines.csv')
existing_pairs = set(
    frozenset((a, b)) for a, b in zip(lines_existing['from_bus'], lines_existing['to_bus'])
)

recoverable = rec[rec['inferred_kv'].notna()].copy()
recoverable['pair'] = recoverable.apply(
    lambda r: frozenset((r['from_bus'], r['to_bus'])), axis=1
)
dupes = recoverable['pair'].isin(existing_pairs)
if dupes.any():
    print(f'Dropping {int(dupes.sum())} duplicates (corridor already in lines.csv)')
recoverable = recoverable[~dupes].copy()

next_id = 0
if len(lines_existing):
    nums = lines_existing['line_id'].str.extract(r'^line_osm_inferred_(\d+)$', expand=False)
    nums = pd.to_numeric(nums, errors='coerce').dropna()
    if len(nums):
        next_id = int(nums.max()) + 1

new_rows = []
for i, r in enumerate(recoverable.itertuples(), start=next_id):
    kv = r.inferred_kv
    if kv not in IMPEDANCE:
        print(f'  warning: no impedance table for {kv} kV, skipping')
        continue
    z = IMPEDANCE[kv]
    new_rows.append({
        'line_id':       f'line_osm_inferred_{i:04d}',
        'from_bus':      r.from_bus,
        'to_bus':        r.to_bus,
        'voltage_kv':    kv,
        'length_km':     r.length_km,
        'r_ohm_per_km':  z['r'],
        'x_ohm_per_km':  z['x'],
        'max_i_ka':      z['i'],
        'is_submarine':  bool(r.is_cable),
        'cable_type':    'submarine' if r.is_cable else 'overhead',
        'is_synthetic':  False,
        'data_source':   'osm_inferred_voltage',
    })
new_df = pd.DataFrame(new_rows, columns=lines_existing.columns)
print(f'\nRecoverable after dedupe: {len(new_df)} lines, {new_df["length_km"].sum():.1f} km total')
if len(new_df):
    print('  by inferred voltage:')
    print(new_df['voltage_kv'].value_counts().sort_index().to_string())

Dropping 7 duplicates (corridor already in lines.csv)

Recoverable after dedupe: 6 lines, 131.7 km total
  by inferred voltage:
voltage_kv
60.0     1
138.0    4
350.0    1


## §4 — Append + verify

In [5]:
before_lines = len(lines_existing)
lines_new = pd.concat([lines_existing, new_df], ignore_index=True)

# Schema sanity
assert list(lines_new.columns) == list(lines_existing.columns)
assert lines_new['line_id'].is_unique, 'duplicate line_id after append'
valid_bus = set(buses['bus_id'])
missing_from = ~lines_new['from_bus'].isin(valid_bus)
missing_to   = ~lines_new['to_bus'].isin(valid_bus)
assert not missing_from.any(), f'{missing_from.sum()} new lines reference unknown from_bus'
assert not missing_to.any(), f'{missing_to.sum()} new lines reference unknown to_bus'

lines_new.to_csv(PROC / 'lines.csv', index=False)
print(f'Wrote lines.csv: {before_lines} → {len(lines_new)} rows (+{len(lines_new) - before_lines})')

Wrote lines.csv: 2409 → 2415 rows (+6)


## §5 — Connectivity impact (transmission subgraph)

Component count on the *transmission* sub-graph (buses with `voltage_kv ≥ 33`
and `bus_type ∈ {substation, tower, hvdc}`), before vs after. Per-province
touch count for the recovered lines so we can see where the fix lands.

In [6]:
tx_ids = set(tx['bus_id'])

def components(df):
    G = nx.Graph()
    G.add_nodes_from(tx_ids)
    for _, ln in df.iterrows():
        if ln['from_bus'] in tx_ids and ln['to_bus'] in tx_ids:
            G.add_edge(ln['from_bus'], ln['to_bus'])
    return nx.number_connected_components(G)

c_before = components(lines_existing)
c_after  = components(lines_new)
print(f'Transmission components BEFORE: {c_before}')
print(f'Transmission components AFTER:  {c_after}')
print(f'Δ components:                   {c_before - c_after}')

if len(new_df):
    bus_prov = dict(zip(buses['bus_id'], buses['province']))
    province_hits = pd.concat([
        new_df['from_bus'].map(bus_prov),
        new_df['to_bus'].map(bus_prov),
    ]).value_counts()
    print('\nRecovered lines by province (counting each endpoint):')
    print(province_hits.to_string())

Transmission components BEFORE: 67
Transmission components AFTER:  64
Δ components:                   3

Recovered lines by province (counting each endpoint):
Cebu              4
Leyte             2
Samar             2
Southern Leyte    2
Eastern Samar     2
